In [1]:
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
from stable_baselines3.common.vec_env import DummyVecEnv

import src.config as config
from src.fed_env import FedEnvBase
from src.ppo import PPOAgent

In [2]:
class MockActorCritic(nn.Module):
    """
    a tiny fake network to test the ppo loop data flow
    """
    def __init__(self, obs_dim, act_dim):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.ReLU(),
            nn.Linear(32, act_dim)
        )
        self.critic = nn.Sequential(
            nn.Linear(obs_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def get_action_and_value(self, obs):
        logits = self.actor(obs)
        probs = Categorical(logits=logits)
        action = probs.sample()

        return action, probs.log_prob(action), self.critic(obs)

    def evaluate_actions(self, obs, action):
        logits = self.actor(obs)
        probs = Categorical(logits=logits)

        return probs.log_prob(action), probs.entropy(), self.critic(obs)

In [4]:
# setup the vectorized environment exactly like train.py
device = torch.device("cpu")
envs = DummyVecEnv([lambda: FedEnvBase(llm_dim=config.LLM_DIM) for _ in range(config.N_ENVS)])

# calculate dimensions
obs_dim = 3 + config.LLM_DIM
act_dim = envs.action_space.n

# init the mock network
fake_network = MockActorCritic(obs_dim, act_dim).to(device)

# init manual ppo agent
agent = PPOAgent(envs, fake_network, device)
total_timesteps_to_test = config.N_STEPS * config.N_ENVS * 2

try:
    agent.learn(total_timesteps=total_timesteps_to_test)
    print("PPO works")
except Exception as e:
    print(f"Something is broken :(")

PPO works
